# 🖼️ 03_precompute_and_tfrecords — Resize, Clean, Prepare 📦✨
Purpose:
- Resize all images referenced in `metadata_with_kp.csv` (or `metadata_rebuilt_mapped.csv`) to **224×224**.
- Store resized images in `data_sih/processed/images_224/`.
- Create `metadata_final_precomputed.csv` with `image_224` column for easy training.
- Optional: produce TFRecords for fast TF training.

Run in order. Keep calm and resize fish. 🧼🐠


## Cell 1 — Imports & paths (Code)

In [37]:
# Cell 1
from pathlib import Path
import pandas as pd, os, cv2, numpy as np
ROOT = Path("data_sih")
PROC = ROOT/"processed"
meta_candidates = [PROC/"metadata_with_kp.csv", PROC/"metadata_rebuilt_mapped.csv", PROC/"metadata_rebuilt_raw.csv"]
meta_path = next((p for p in meta_candidates if p.exists()), None)
assert meta_path is not None, "No metadata found. Run Notebook 01/02."
df = pd.read_csv(meta_path)
print("Using metadata:", meta_path.name, "| rows:", len(df))


Using metadata: metadata_with_kp.csv | rows: 12064


## Cell 2 — Setup output folder & constants (Code)

In [38]:
# Cell 2
IMG_SIZE = 224
OUT_DIR = PROC/"images_224"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_META = PROC/"metadata_final_precomputed.csv"
print("Out dir:", OUT_DIR)


Out dir: data_sih\processed\images_224


## Cell 3 — Robust resize helper (Code)

In [39]:
# Cell 3
def resize_and_save(src_path, out_path, size=IMG_SIZE, quality=90):
    try:
        if not isinstance(src_path,str) or not os.path.exists(src_path):
            return False
        img = cv2.imread(src_path)
        if img is None:
            return False
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (size,size), interpolation=cv2.INTER_AREA)
        ok = cv2.imwrite(str(out_path), cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        return bool(ok)
    except Exception:
        return False


## Cell 4 — Resize loop: create images_224 and add image_224 field (Code)

In [40]:
# Cell 4
if 'image_224' not in df.columns:
    df['image_224'] = ""

resized = 0; skipped = 0; failed = []

for idx,row in df.iterrows():
    src = row.get('image_path','')
    if not isinstance(src,str) or not os.path.exists(src):
        failed.append((idx,src)); continue
    out_name = Path(src).stem + f"_{idx}.jpg"
    out_path = OUT_DIR/out_name
    if out_path.exists():
        df.at[idx,'image_224'] = str(out_path.resolve()); skipped += 1; continue
    ok = resize_and_save(src, out_path)
    if ok:
        df.at[idx,'image_224'] = str(out_path.resolve()); resized += 1
    else:
        failed.append((idx,src))

print(f"Resized: {resized} | Skipped: {skipped} | Failed: {len(failed)}")
if failed:
    print("Sample failures:", failed[:20])
df.to_csv(OUT_META, index=False)
print("Saved final metadata:", OUT_META)


Resized: 3064 | Skipped: 9000 | Failed: 0
Saved final metadata: data_sih\processed\metadata_final_precomputed.csv


## Cell 5 — Optional: TFRecord builder (Markdown + Code snippet)

In [41]:
# Cell 5 — optional TFRecord builder (disabled by default)
MAKE_TFRECORDS = True
if MAKE_TFRECORDS:
    import tensorflow as tf
    TFREC_DIR = PROC/"tfrecords"; TFREC_DIR.mkdir(exist_ok=True)
    tfrec_path = TFREC_DIR/"data.tfrecord"
    def _bytes_feature(v): return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
    def _float_feature(v): return tf.train.Feature(float_list=tf.train.FloatList(value=[v]))
    def _int_feature(v): return tf.train.Feature(int64_list=tf.train.Int64List(value=[v]))
    with tf.io.TFRecordWriter(str(tfrec_path)) as wr:
        for _,r in df.iterrows():
            img_p = r['image_224']
            if not (isinstance(img_p,str) and os.path.exists(img_p)): continue
            raw = tf.io.read_file(img_p).numpy()
            feature = {
                "image": _bytes_feature(raw),
                "freshness": _float_feature(float(r.get('freshness', -1.0) if not pd.isna(r.get('freshness')) else -1.0)),
                "head_x": _float_feature(float(r.get('head_x', -1.0) if not pd.isna(r.get('head_x')) else -1.0)),
                "head_y": _float_feature(float(r.get('head_y', -1.0) if not pd.isna(r.get('head_y')) else -1.0)),
                "tail_x": _float_feature(float(r.get('tail_x', -1.0) if not pd.isna(r.get('tail_x')) else -1.0)),
                "tail_y": _float_feature(float(r.get('tail_y', -1.0) if not pd.isna(r.get('tail_y')) else -1.0))
            }
            ex = tf.train.Example(features=tf.train.Features(feature=feature))
            wr.write(ex.SerializeToString())
    print("Wrote TFRecord:", tfrec_path)


Wrote TFRecord: data_sih\processed\tfrecords\data.tfrecord


## Cell 6 — Sanity checks & summary (Code)

In [42]:
# Cell 6
df = pd.read_csv(OUT_META)
from pathlib import Path
df['exists_224'] = df['image_224'].apply(lambda p: isinstance(p,str) and Path(p).exists())
print("Total rows:", len(df))
print("Exists by source:\n", df.groupby('source')['exists_224'].sum())
print("Species unique:", df['species'].nunique())
print("Rows with freshness labels:", df['freshness'].notna().sum())
print("Rows with keypoints:", df['head_x'].notna().sum())
display(df[df['source']=='species'].head(3)[['image_path','image_224','species']])
display(df[df['source']=='disease'].head(3)[['image_path','image_224','disease_label','freshness']])
display(df[df['source']=='deepfish'].head(3)[['image_path','image_224','mask_path','head_x','tail_x']])


Total rows: 12064
Exists by source:
 source
deepfish     620
disease     2444
species     9000
Name: exists_224, dtype: int64
Species unique: 10
Rows with freshness labels: 2444
Rows with keypoints: 310


,image_path,image_224,species
0,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Black Sea Sprat
1,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Black Sea Sprat
2,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Black Sea Sprat


,image_path,image_224,disease_label,freshness
9000,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Bacterial diseases - Aeromoniasis,0.2
9001,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Bacterial diseases - Aeromoniasis,0.2
9002,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,Bacterial diseases - Aeromoniasis,0.2


,image_path,image_224,mask_path,head_x,tail_x
11444,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,NaN,NaN
11445,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,NaN,NaN
11446,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,C:\Users\Hp\Project\FishNet-ML-Grand-Finale\da...,NaN,NaN


# Troubleshooting quick guide 🩺

- If `Exists counts by source` shows zeros for a source:
  - Check `image_path` column for that source (maybe wrong paths).
  - Inspect the `failed` list printed in Cell 4 for specific failing `image_path`s.
  - Use the resize helper manually on a single failing path to see why OpenCV returns None.

- If many files failed with weird characters or very long Windows paths:
  - Move dataset to a shorter path (e.g. C:\data_sih) or normalize filenames.

- If masks were polygons in CSV (no mask PNGs):
  - Tell me and I’ll give a small cell to render polygons to binary PNG masks (I saw `segmentation.csv` earlier).

- After confirming all sources exist in `metadata_final_precomputed.csv`, run Notebook 04 (training).
